# POC: Hebrew Title Extraction Fine-Tuning

This notebook is a minimal proof-of-concept for TODO section A:

- Download `biunlp/HeSum`
- Download a small Qwen model
- Fine-tune on a tiny subset
- Generate a few predicted Hebrew titles

The goal is not quality yet. The goal is to verify that the full training path runs end-to-end.

## 0. Install dependencies

Run this once in the notebook environment. In Colab, avoid upgrading `torch` because it can replace the preinstalled CUDA stack and create dependency conflicts. The cell also removes Colab's old `torchao`, which can conflict with recent PEFT. If package versions changed, restart the runtime before continuing.

In [ ]:
# Do not upgrade torch/CUDA in Colab; use the preinstalled GPU stack.
# Install only missing project dependencies.
%pip install -q "datasets" "accelerate" "peft" "evaluate" "rouge-score" "sentencepiece"

# Colab may include an old torchao version that conflicts with recent PEFT.
# LoRA fine-tuning here does not need torchao.
%pip uninstall -y -q torchao

## 0.1. Recovery if the old install cell broke Colab

If you previously ran a cell that upgraded `torch`, the Colab runtime can become polluted with mismatched CUDA, PyTorch, NumPy, and `fsspec` packages. Do not try to repair this in-place.

Use Colab: **Runtime -> Disconnect and delete runtime**, then reconnect and run from the top. This is the reliable fix.

In [ ]:
# Local/Cursor users can ignore this cell.
# In Colab, if a previous run polluted CUDA packages, use Runtime -> Disconnect and delete runtime.
print("Recovery note acknowledged; continuing.")

## 1. Imports and configuration

Keep the POC tiny. Increase `MAX_STEPS`, `TRAIN_SAMPLES`, and `MAX_LENGTH` only after this runs successfully.

In [ ]:
import os
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_NAME = "biunlp/HeSum"
OUTPUT_DIR = Path("outputs/poc-qwen-hesum")

TRAIN_SAMPLES = 100
EVAL_SAMPLES = 5
MAX_LENGTH = 1024
MAX_STEPS = 20

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

## 2. Download and inspect the dataset

This uses the Hugging Face dataset id `biunlp/HeSum`. The original project/GitHub repo is `OnlpLab/HeSum`, but `load_dataset` needs the Hugging Face dataset id. This cell also verifies the actual column names before training.

In [ ]:
raw_datasets = load_dataset(DATASET_NAME)
raw_datasets

In [ ]:
split_name = "train" if "train" in raw_datasets else list(raw_datasets.keys())[0]
sample = raw_datasets[split_name][0]

print("Using split:", split_name)
print("Columns:", raw_datasets[split_name].column_names)
sample

## 3. Convert HeSum examples into instruction-tuning text

The helper below tries common column names because dataset schemas sometimes differ between versions. If it raises an error, inspect the printed columns above and update `ARTICLE_CANDIDATES` / `TITLE_CANDIDATES`.

In [ ]:
ARTICLE_CANDIDATES = ["article", "text", "content", "body", "document"]
TITLE_CANDIDATES = ["summary", "title", "sub_heading", "subheading", "headline"]

def first_existing(example, candidates):
    for key in candidates:
        value = example.get(key)
        if isinstance(value, str) and value.strip():
            return key, value.strip()
    raise KeyError(f"None of these columns were found with non-empty text: {candidates}. Available columns: {list(example.keys())}")

def build_prompt(article):
    return (
        "\u05d0\u05ea\u05d4 \u05e2\u05d5\u05d6\u05e8 \u05de\u05d7\u05e7\u05e8 \u05d1\u05e2\u05d9\u05d1\u05d5\u05d3 \u05e9\u05e4\u05d4 \u05d8\u05d1\u05e2\u05d9\u05ea.\n"
        "\u05e7\u05e8\u05d0 \u05d0\u05ea \u05d4\u05db\u05ea\u05d1\u05d4 \u05d4\u05d1\u05d0\u05d4 \u05d5\u05db\u05ea\u05d5\u05d1 \u05db\u05d5\u05ea\u05e8\u05ea \u05e7\u05e6\u05e8\u05d4 \u05d5\u05de\u05d3\u05d5\u05d9\u05e7\u05ea \u05d1\u05e2\u05d1\u05e8\u05d9\u05ea.\n\n"
        f"\u05db\u05ea\u05d1\u05d4:\n{article}\n\n"
        "\u05db\u05d5\u05ea\u05e8\u05ea:"
    )

def format_example(example):
    _, article = first_existing(example, ARTICLE_CANDIDATES)
    _, title = first_existing(example, TITLE_CANDIDATES)
    prompt = build_prompt(article)
    return {
        "prompt": prompt,
        "title": title,
        "text": f"{prompt} {title}",  # Kept only for inspection/debugging.
    }

formatted = raw_datasets[split_name].select(range(min(TRAIN_SAMPLES, len(raw_datasets[split_name])))).map(format_example)
print(formatted[0]["text"][:1200])

## 4. Download model and tokenizer

This uses Qwen 0.5B for a fast POC. If memory is tight, reduce `MAX_LENGTH` first.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None
model.to(device)
print(f"Loaded {MODEL_NAME} on {device}")

## 5. Tokenize the tiny training subset

The labels mask the prompt with `-100`, so the loss is computed only on the gold title. This gives the LoRA adapter a much cleaner learning signal.

In [ ]:
def tokenize_batch(batch):
    input_ids_batch = []
    attention_mask_batch = []
    labels_batch = []

    for prompt, title in zip(batch["prompt"], batch["title"]):
        answer = " " + title.strip()
        if tokenizer.eos_token:
            answer += tokenizer.eos_token

        # Tokenize prompt and answer separately so truncation never removes the title.
        answer_tokens = tokenizer(answer, add_special_tokens=False, padding=False)
        answer_ids = answer_tokens["input_ids"][:MAX_LENGTH]
        max_prompt_length = max(1, MAX_LENGTH - len(answer_ids))

        prompt_tokens = tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=True,
            max_length=max_prompt_length,
            padding=False,
        )
        prompt_ids = prompt_tokens["input_ids"]

        remaining_length = MAX_LENGTH - len(prompt_ids)
        answer_ids = answer_ids[:remaining_length]
        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids.copy()

        input_ids_batch.append(input_ids)
        attention_mask_batch.append([1] * len(input_ids))
        labels_batch.append(labels)

    return {
        "input_ids": input_ids_batch,
        "attention_mask": attention_mask_batch,
        "labels": labels_batch,
    }

tokenized = formatted.map(tokenize_batch, batched=True, remove_columns=formatted.column_names)
label_lengths = [sum(label != -100 for label in labels) for labels in tokenized["labels"]]
print(f"Average supervised title tokens: {sum(label_lengths) / len(label_lengths):.1f}")
tokenized

## 6. Add LoRA adapters

Only a small number of trainable parameters are updated, which is enough for the POC.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Tiny fine-tuning run

Success criterion: the training loop starts, logs loss, and writes a checkpoint.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=MAX_STEPS,
    learning_rate=2e-4,
    logging_steps=1,
    save_steps=MAX_STEPS,
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

# Preserves our custom labels, including -100 over prompt tokens.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    padding=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR / "final-adapter"))

## 8. Generate a few sanity-check titles

This is not evaluation yet. It just confirms that the trained adapter can run inference.

In [ ]:
model.eval()

def generate_title(example, max_new_tokens=96):
    _, article = first_existing(example, ARTICLE_CANDIDATES)
    _, gold_title = first_existing(example, TITLE_CANDIDATES)
    prompt = build_prompt(article)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return {"gold": gold_title, "prediction": generated}

for i in range(min(EVAL_SAMPLES, len(raw_datasets[split_name]))):
    result = generate_title(raw_datasets[split_name][i])
    print(f"--- Example {i} ---")
    print("Gold:", result["gold"])
    print("Pred:", result["prediction"])


## Next steps after the POC works

- Split the dataset into full / lead-only / body-only variants.
- Add ROUGE and BERTScore evaluation.
- Increase sample count and training steps.
- Save generated predictions to `outputs/` for manual or LLM-as-judge evaluation.